In [1]:
!pip install einops
!pip install datasets
!pip install sentencepiece

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from einops import rearrange
import random
import numpy as np

In [2]:
class CharCNNEncoder(nn.Module):
    def __init__(self, char_vocab_size, char_emb_dim, out_channels, kernel_sizes, max_word_len):
        super().__init__()
        self.char_emb = nn.Embedding(char_vocab_size, char_emb_dim)
        self.convs = nn.ModuleList([
            nn.Conv2d(1, out_ch, (k, char_emb_dim))
            for k, out_ch in zip(kernel_sizes, out_channels)
        ])
        self.max_word_len = max_word_len

    def forward(self, x):
        # x: (batch_size, max_seq_len, max_word_len)
        batch_size, seq_len, word_len = x.size()
        x = self.char_emb(x)  # (B, S, W, E)
        x = rearrange(x, 'b s w e -> (b s) 1 w e')  # reshape for conv
        conv_outs = [F.relu(conv(x)).squeeze(3) for conv in self.convs]
        pooled = [F.max_pool1d(out, out.size(2)).squeeze(2) for out in conv_outs]
        rep = torch.cat(pooled, dim=1)  # (B * S, sum(out_channels))
        rep = rep.view(batch_size, seq_len, -1)
        return rep  # shape: (batch_size, seq_len, representation_dim)

In [3]:
class Highway(nn.Module):
    def __init__(self, size, num_layers=1):
        super().__init__()
        self.num_layers = num_layers
        self.nonlinear = nn.ModuleList([nn.Linear(size, size) for _ in range(num_layers)])
        self.gate = nn.ModuleList([nn.Linear(size, size) for _ in range(num_layers)])

    def forward(self, x):
        for i in range(self.num_layers):
            H = F.relu(self.nonlinear[i](x))
            T = torch.sigmoid(self.gate[i](x))
            x = H * T + x * (1 - T)
        return x

In [4]:
class CharAwareLM(nn.Module):
    def __init__(self, char_vocab_size, char_emb_dim, cnn_out_channels, kernel_sizes,
                 max_word_len, highway_layers, hidden_dim, vocab_size, num_layers=1):
        super().__init__()
        self.char_encoder = CharCNNEncoder(char_vocab_size, char_emb_dim, cnn_out_channels, kernel_sizes, max_word_len)
        cnn_output_size = sum(cnn_out_channels)

        self.highway = Highway(cnn_output_size, num_layers=highway_layers)
        self.lstm = nn.LSTM(input_size=cnn_output_size, hidden_size=hidden_dim,
                            num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x_char, hidden=None):
        # x_char: (batch_size, seq_len, max_word_len)
        rep = self.char_encoder(x_char)
        rep = self.highway(rep)
        out, hidden = self.lstm(rep, hidden)
        logits = self.fc(out)  # (batch, seq_len, vocab)
        return logits, hidden

In [5]:
!git clone https://github.com/yoonkim/lstm-char-cnn.git
%cd lstm-char-cnn

fatal: destination path 'lstm-char-cnn' already exists and is not an empty directory.
/content/lstm-char-cnn


In [8]:
!pip install datasets --quiet
!pip install torch==1.13.1 --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 15.5 MB/s eta 0:00:00


In [9]:
from datasets import load_dataset

dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

# Save to plain text files
with open("train.txt", "w") as f:
    f.write("\n".join(dataset["train"]["text"]))

with open("valid.txt", "w") as f:
    f.write("\n".join(dataset["validation"]["text"]))

with open("test.txt", "w") as f:
    f.write("\n".join(dataset["test"]["text"]))

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/733k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/6.36M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

In [10]:
!mkdir -p data
!mv train.txt valid.txt test.txt data/

In [12]:
!python main.py --data data --cuda --epochs 50 --batch_size 20

python3: can't open file '/content/lstm-char-cnn/main.py': [Errno 2] No such file or directory
